# Logistics Dataset Generation
This notebook generates a synthetic logistics dataset for delivery and carrier performance analysis. It includes delivery timings, shipment costs, customer types, and carrier performance metrics.


## 1. Install dependencies
Use this cell if the required packages are not already installed in your environment.


In [ ]:
# Install required packages if needed
!pip install pandas numpy


## 2. Imports and random seed
Import required libraries and set deterministic randomness for reproducible dataset generation.


In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

# عدد السجلات
n = 3000


## 3. Scenario definitions
Define regions, cities, shipment types, customer segments, carriers, and product categories used in the synthetic dataset.


In [ ]:
regions = {
    "Central": ["Riyadh", "Buraydah"],
    "Western": ["Jeddah", "Makkah", "Madinah"],
    "Eastern": ["Dammam", "Khobar", "Jubail"],
    "Northern": ["Tabuk", "Hail"],
    "Southern": ["Abha", "Jizan", "Najran"]
}

shipment_types = ["Standard", "Express", "Same-Day"]
shipment_probs = [0.6, 0.3, 0.1]

customer_types = ["Retail", "Business", "VIP"]
customer_probs = [0.7, 0.2, 0.1]

carriers = ["FastGo", "PrimeRoute", "AramexPlus", "LocalShip"]

product_categories = ["Electronics", "Fashion", "Grocery", "Home", "Medical"]


## 4. Helper functions and delay factors
Create reusable functions to sample dates, compute distances, assign weights, and estimate order values. Also define carrier and region delay factors for delivery performance modeling.


In [ ]:
def random_date(start, end):
    """Return a random date between start and end."""
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

def get_distance(region):
    """Estimate shipping distance by region."""
    if region in ["Central", "Western", "Eastern"]:
        return np.random.randint(20, 500)
    return np.random.randint(100, 900)

def get_weight():
    """Return a shipment weight, with most shipments under 12 kg."""
    if np.random.rand() < 0.8:
        return round(np.random.uniform(0.5, 12), 2)
    return round(np.random.uniform(12, 30), 2)

def get_order_value(category):
    """Estimate the order value based on product category."""
    if category == "Electronics":
        return round(np.random.uniform(500, 5000), 2)
    elif category == "Fashion":
        return round(np.random.uniform(50, 500), 2)
    elif category == "Grocery":
        return round(np.random.uniform(20, 200), 2)
    return round(np.random.uniform(100, 1500), 2)

def get_base_delivery_days(shipment_type):
    """Return the expected delivery window for each shipment type."""
    if shipment_type == "Standard":
        return np.random.randint(3, 7)
    if shipment_type == "Express":
        return np.random.randint(1, 4)
    return np.random.randint(0, 2)

carrier_delay_factor = {
    "FastGo": -0.5,
    "PrimeRoute": 0,
    "AramexPlus": 0.5,
    "LocalShip": 1
}

region_delay_factor = {
    "Central": 0,
    "Western": 0.2,
    "Eastern": 0.3,
    "Northern": 0.7,
    "Southern": 0.8
}


## 5. Generate synthetic shipment records
Loop through the required number of rows and generate shipment details, delivery dates, cost, status, returns, and customer ratings.


In [ ]:
data = []
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)

for i in range(n):
    region = random.choice(list(regions.keys()))
    city = random.choice(regions[region])
    shipment_type = np.random.choice(shipment_types, p=shipment_probs)
    customer_type = np.random.choice(customer_types, p=customer_probs)
    carrier = random.choice(carriers)
    category = random.choice(product_categories)

    order_date = random_date(start_date, end_date)
    dispatch_date = order_date + timedelta(days=np.random.randint(0, 3))
    base_days = get_base_delivery_days(shipment_type)

    distance = get_distance(region)
    weight = get_weight()

    delay_score = 0
    delay_score += distance / 500
    delay_score += weight / 20
    delay_score += region_delay_factor[region] + carrier_delay_factor[carrier]

    if shipment_type == "Express":
        delay_score -= 0.5
    elif shipment_type == "Same-Day":
        delay_score -= 0.3
    if customer_type == "VIP":
        delay_score -= 0.3

    if delay_score < 1.5:
        delay_days = 0
    elif delay_score < 2.5:
        delay_days = np.random.randint(1, 3)
    else:
        delay_days = np.random.randint(2, 6)

    promised_delivery_date = dispatch_date + timedelta(days=base_days)
    delivery_date = promised_delivery_date + timedelta(days=delay_days)

    cost = 10 + distance * 0.05 + weight * 0.8
    if shipment_type == "Express":
        cost *= 1.3
    elif shipment_type == "Same-Day":
        cost *= 1.6
    if region in ["Northern", "Southern"]:
        cost *= 1.2

    status = "Delivered" if delay_days == 0 else "Delayed"
    returned = False
    if delay_days > 3 and np.random.rand() < 0.3:
        returned = True
        status = "Returned"

    if status == "Returned":
        rating = np.random.randint(1, 3)
    elif delay_days == 0:
        rating = np.random.randint(4, 6)
    elif delay_days <= 2:
        rating = np.random.randint(3, 5)
    else:
        rating = np.random.randint(1, 3)

    order_value = get_order_value(category)

    data.append([
        i + 1, order_date, dispatch_date, promised_delivery_date, delivery_date,
        region, city, customer_type, shipment_type, carrier, category, weight, distance,
        round(cost, 2), order_value, status, delay_days, rating
    ])


## 6. Create a DataFrame and add derived features
Convert the generated records into a Pandas DataFrame and compute extra features that are useful for analysis.


In [ ]:
columns = [
    "shipment_id", "order_date", "dispatch_date",
    "promised_delivery_date", "delivery_date",
    "region", "city", "customer_type",
    "shipment_type", "carrier", "product_category",
    "weight_kg", "distance_km", "shipping_cost",
    "order_value", "delivery_status", "delay_days",
    "customer_rating"
]

df = pd.DataFrame(data, columns=columns)

df["delivery_time_days"] = (df["delivery_date"] - df["dispatch_date"]).dt.days
df["on_time_flag"] = (df["delay_days"] == 0).astype(int)
df["cost_per_km"] = df["shipping_cost"] / df["distance_km"]
df["cost_per_kg"] = df["shipping_cost"] / df["weight_kg"]


## 7. Save the dataset
Persist the generated dataset to a CSV file for later analysis.


In [ ]:
output_path = "logistics_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Dataset created successfully: {output_path}")
